# Bangladesh: 0226_climate_bootcamp_session9

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0226_climate_bootcamp_session9
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'soil_drought' / '0226_climate_bootcamp_session9'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""09_class9_drought_risk_mapping.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Climate Data Analysis Bootcamp using GEE and Python - Session9: Drought Risk Mapping
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor

"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map()
map
roi = BD_ROI
roi
'# **Drought Image Collection**'
terra = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').filterDate('2010', '2020').select('pdsi', 'aet', 'pet', 'pr', 'ro', 'soil', 'tmmn', 'tmmx')
terra
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
grid_20km = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:4326', grid_scale=(0.2, -0.2))
ds = xr.open_dataset(terra, engine='ee', **grid_20km)
ds = ds.sortby('time') * 1
ds['pdsi_class'] = (ds['pdsi'] * 0.01 < -2.0).astype(int)
ds = ds.drop_vars('pdsi')
ds
'# **Sampling**'
df = ds.to_dataframe().dropna()
x = df.drop('pdsi_class', axis=1)
x
y = df['pdsi_class']
y
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
'# **Modelling**'
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
y_pred = model.predict_proba(x_test)[:, 1]
from sklearn.metrics import roc_auc_score
print(roc_auc_score(y_test, y_pred))
'# **Risk Mapping**'
df['drought_risk'] = model.predict_proba(x)[:, 1]
risk_map = df['drought_risk'].to_xarray()
risk_map_mean = risk_map.mean(dim='time')
risk_map_mean.plot.contourf(x='x', y='y', robust=True, cmap='turbo', levels=10)
'# **Export**'
import rioxarray
out = risk_map_mean.copy()
out = out.rio.write_crs('EPSG:4326')
out = out.rio.set_spatial_dims(x_dim='x', y_dim='y')
out = out.rio.write_coordinate_system(inplace=True)
out
out.rio.to_raster('drought_risk.tiff')


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'soil_drought', '0226_climate_bootcamp_session9', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'soil_drought')  # explicit upload; requires Drive OAuth scope
